In [ ]:
import torchvision
print(torchvision.__version__)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import time 

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define transformations for the STL-10 dataset
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize images to 128x128
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Load STL-10 dataset
train_dataset = STL10(root="path to dataset", split="train", download=False, transform=transform)
test_dataset = STL10(root="path to dataset", split="test", download=False, transform=transform)

# Select only the first 50 samples from the train dataset
train_dataset_200 = torch.utils.data.Subset(train_dataset, range(200))

# Create DataLoaders
train_loader = DataLoader(train_dataset_200, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Display a few sample images
def show_samples(dataset, num_samples=4):
    plt.figure(figsize=(10, 5))
    for i in range(num_samples):
        image, label = dataset[i]
        image = image.permute(1, 2, 0)  # Change to HxWxC format for display
        image = (image * 0.5 + 0.5).numpy()  # Unnormalize
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(image)
        plt.title(f"Label: {label}")
        plt.axis("off")
    plt.show()

# Display the first 4 samples from the 50 selected samples
show_samples(train_dataset_200, num_samples=4)

# Define the RGB to Grayscale function
def rgb2grey_py(x):
################################################################################################
#WRITE YOUR PYHTON CODE HERER
###############################################################################################

# Convert all images in the first 50 samples to grayscale
grayscale_images = []
start_time_python = time.time()
for i, (image, label) in enumerate(train_dataset_200):
    image = image.to(device)  # Send to the same device as tensor operations
    gray_image = rgb2grey_py(image)
    grayscale_images.append((gray_image, label))

end_time_python = time.time()
python_time = end_time_python - start_time_python
print(f"Time taken for Python-based grayscale conversion: {python_time:.4f} seconds")

# Plot the first 4 grayscale images
plt.figure(figsize=(10, 5))
for i in range(4):
    original_image, label = train_dataset_200[i]
    original_image = original_image.permute(1, 2, 0).cpu().numpy()  # Convert to HxWxC
    original_image = (original_image * 0.5 + 0.5)  # Unnormalize

    gray_image, label = grayscale_images[i]
    gray_image = gray_image.cpu().numpy()

    plt.subplot(1, 4, i + 1)
    plt.imshow(gray_image, cmap="gray")
    plt.title(f"Grayscale Image {i + 1}")
    plt.axis("off")

plt.show()


In [ ]:
%pip install -q wurlitzer ninja

In [ ]:
%load_ext wurlitzer

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from torch import tensor
import torchvision as tv
import torchvision.transforms.functional as tvf
from torchvision import io
from torch.utils.cpp_extension import load_inline

# Setting up environment variable
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Function to load CUDA kernel
def load_cuda(cuda_src, cpp_src, funcs, opt=False, verbose=False):
    return load_inline(cuda_sources=[cuda_src], cpp_sources=[cpp_src], functions=funcs,
                       extra_cuda_cflags=["-O3"] if opt else [], verbose=verbose, name="inline_ext")

# CUDA and C++ source code
cuda_begin = r'''
#include <torch/extension.h>
#include <stdio.h>
#include <c10/cuda/CUDAException.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

inline unsigned int cdiv(unsigned int a, unsigned int b) { return (a + b - 1) / b;}
'''

cuda_src = cuda_begin + r'''
#########################################################################
#WRITE YOUR CUDA KERNEL
#########################################################################

'''

cpp_src = "torch::Tensor rgb_to_grayscale(torch::Tensor input);"

# Compile and load the CUDA kernel
module = load_cuda(cuda_src, cpp_src, funcs=["rgb_to_grayscale"])

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


grayscale_images = []

start_time_python = time.time()
for i, (image, label) in enumerate(train_dataset_200):
    image = image.to(device)  # Send to the same device as tensor operations
    
    # Un-normalize the image from [-1, 1] to [0, 255]
    image = (image + 1) * 127.5  # Convert from [-1, 1] to [0, 255]
    image = image.clamp(0, 255)  # Ensure values are in the valid range [0, 255]
    
    # Convert to uint8 and move to GPU (for CUDA kernel)
    image_uint8 = image.to(torch.uint8).contiguous().to(device)
    
    # Call the CUDA kernel for grayscale conversion
    gray_image = module.rgb_to_grayscale(image_uint8)  # Pass the uint8 image to the kernel
    
    # Append the grayscale image and label
    grayscale_images.append((gray_image, label))

end_time_python = time.time()
python_time = end_time_python - start_time_python
print(f"Time taken for Python-based grayscale conversion: {python_time:.4f} seconds")

# Plot the first 4 grayscale images
plt.figure(figsize=(10, 5))
for i in range(4):
    original_image, label = train_dataset_200[i]
    original_image = original_image.permute(1, 2, 0).cpu().numpy()  # Convert to HxWxC
    original_image = (original_image * 0.5 + 0.5)  # Unnormalize to display the original image

    gray_image, label = grayscale_images[i]
    gray_image = gray_image.cpu().numpy()

    plt.subplot(1, 4, i + 1)
    plt.imshow(gray_image, cmap="gray")
    plt.title(f"Grayscale Image {i + 1}")
    plt.axis("off")

plt.show()

